<a href="https://colab.research.google.com/github/stefkong1982/netology.ru/blob/Master/%D0%9E%D0%B1%D1%83%D1%87%D0%B0%D1%8E%D1%89%D0%B8%D0%B9_%D0%BD%D0%B0%D0%B1%D0%BE%D1%80__%D0%A7%D0%B0%D1%81%D1%82%D1%8C_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Цель работы:**  
Разработать модель, способную предсказывать состав следующего заказа пользователя на основе анализа его истории покупок. Это позволит повысить персонализацию сервиса и улучшить пользовательский опыт, а также оптимизировать процессы формирования корзины и планирования закупок.


# Введение и постановка задачи

Современные сервисы доставки продуктов активно используют данные о покупках пользователей для улучшения качества и персонализации сервиса. Анализ истории заказов позволяет прогнозировать, какие категории товаров пользователь может захотеть приобрести в следующий раз.

Целью данного проекта является создание модели, способной по истории заказов прогнозировать состав будущего заказа пользователя. Такая рекомендация поможет клиенту сэкономить время на формировании корзины, избежать забытых позиций и повысить удобство планирования закупок.

Задача формализована как многоклассовая классификация: необходимо предсказать для каждой пары (пользователь, категория), будет ли категория включена в следующий заказ. Для оценки качества модели используется метрика F1-score, которая учитывает баланс между точностью и полнотой предсказаний.

Данные и постановка задачи основаны на открытом соревновании в области электронных продаж.


# Описание набора данных

В проекте используется история заказов 20 000 пользователей, разделённая на тренировочную и тестовую выборки по дате. Тестовая выборка содержит заказы после определённой даты отсечки.

Основной тренировочный файл содержит следующие данные:  
- **user_id** — уникальный идентификатор пользователя  
- **order_completed_at** — дата и время завершения заказа  
- **cart** — категория товара, входящего в заказ (уникальные категории)

Задача — для каждой пары (пользователь, категория), встречающейся в тестовой выборке, предсказать бинарный признак: будет ли категория присутствовать в следующем заказе пользователя.

Идентификаторы пар представлены в формате `"{user_id};{category_id}"`, что учитывается при обработке данных.

Данные подготовлены на основе истории заказов с учётом особенностей временного разделения выборок.


In [ ]:
# Подключение Google Drive для доступа к данным и сохранения результатов
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Создание структуры проекта
import os
from pathlib import Path
import pandas as pd
import numpy as np


# Основная папка проекта (как вы указали)
project_path = "/content/drive/MyDrive/Colab Notebooks/sm"

# Чек-лист папок и файлов
structure = [
    "data/raw",          # Исходные данные
    "data/processed",    # Очищенные данные
    "notebooks",         # Jupyter-тетради
    "src/models",        # Код моделей
    "src/utils",         # Утилиты
    "scripts",           # Скрипты обработки
]

# Создаём все директории
for folder in structure:
    os.makedirs(os.path.join(project_path, folder), exist_ok=True)


In [ ]:
# Функция для рекурсивного вывода структуры папок проекта
def print_tree(root, prefix=""):
    files = sorted(os.listdir(root))
    for i, name in enumerate(files):
        path = os.path.join(root, name)
        is_last = (i == len(files) - 1)
        branch = "└── " if is_last else "├── "
        print(prefix + branch + name + ("/" if os.path.isdir(path) else ""))
        if os.path.isdir(path):
            new_prefix = prefix + ("    " if is_last else "│   ")
            print_tree(path, new_prefix)

print("\n=== Структура проекта (project_path) ===")
print_tree(project_path)


=== Структура проекта (project_path) ===
├── data/
│   ├── processed/
│   └── raw/
│       ├── sample_submission.csv
│       └── train.csv
├── notebooks/
├── plan/
├── scripts/
└── src/
    ├── models/
    └── utils/


In [ ]:
# Блок: Загрузка и первичный анализ train.csv
import pandas as pd
import os


train = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/sm/data/raw/train.csv")

print("Структура и информация о train.csv:")
print(train.info())

Структура и информация о train.csv:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3123064 entries, 0 to 3123063
Data columns (total 3 columns):
 #   Column              Dtype 
---  ------              ----- 
 0   user_id             int64 
 1   order_completed_at  object
 2   cart                int64 
dtypes: int64(2), object(1)
memory usage: 71.5+ MB
None


In [ ]:
print(f"Уникальных пользователей: {train['user_id'].nunique()}")

Уникальных пользователей: 20000


In [ ]:
train

,user_id,order_completed_at,cart
0,2,2015-03-22 09:25:46,399
1,2,2015-03-22 09:25:46,14
2,2,2015-03-22 09:25:46,198
3,2,2015-03-22 09:25:46,88
4,2,2015-03-22 09:25:46,157
...,...,...,...
3123059,12702,2020-09-03 23:45:45,441
3123060,12702,2020-09-03 23:45:45,92
3123061,12702,2020-09-03 23:45:45,431
3123062,12702,2020-09-03 23:45:45,24


In [ ]:
# Блок: Дополнительная статистика по train.csv

print("\nДополнительная статистика по train.csv:")

orders_per_user = train['user_id'].value_counts()
print("\nРаспределение количества заказов на пользователя:")
print(orders_per_user.describe())

print(f"\nУникальных категорий (корзин): {train['cart'].nunique()}")

train['order_completed_at'] = pd.to_datetime(train['order_completed_at'])
print("\nСтатистика по датам заказов:")
print(f"Период с {train['order_completed_at'].min()} по {train['order_completed_at'].max()}")



Дополнительная статистика по train.csv:

Распределение количества заказов на пользователя:
count    20000.000000
mean       156.153200
std        200.840781
min          3.000000
25%         48.000000
50%         88.000000
75%        181.000000
max       3508.000000
Name: count, dtype: float64

Уникальных категорий (корзин): 881

Статистика по датам заказов:
Период с 2015-03-22 09:25:46 по 2020-09-03 23:45:45


In [ ]:
# Блок: Загрузка и первичный анализ sample_submission.csv
sub = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/sm/data/raw/sample_submission.csv")

print("Структура и информация:")
print(sub.info())

Структура и информация:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 790449 entries, 0 to 790448
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   id      790449 non-null  object
 1   target  790449 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 12.1+ MB
None


In [ ]:
sub

,id,target
0,0;133,0
1,0;5,1
2,0;10,0
3,0;396,1
4,0;14,0
...,...,...
790444,19998;26,0
790445,19998;31,0
790446,19998;29,1
790447,19998;798,1


In [ ]:
# Блок: Агрегация корзин и добавление порядкового номера заказа
#
# Назначение:
#   - Сгруппировать данные по пользователям и времени завершения заказа.
#   - Собрать список категорий товаров (корзину) в каждом заказе.
#   - Добавить порядковый номер заказа для каждого пользователя.
#
# Вход:
#   - DataFrame train с данными заказов.
# Выход:
#   - DataFrame orders_agg с агрегированными корзинами и порядковыми номерами.
#
train['order_completed_at'] = pd.to_datetime(train['order_completed_at'])

orders_agg = (
    train
    .groupby(['user_id', 'order_completed_at'])['cart']
    .apply(list)
    .reset_index()
    .rename(columns={'cart': 'cart_list'})
    .sort_values(['user_id', 'order_completed_at'])
)

orders_agg['order_number'] = (
    orders_agg
    .groupby('user_id')
    .cumcount() + 1
)

user_last_order = (
    orders_agg.groupby('user_id')['order_number'].max()
    .reset_index()
    .rename(columns={'order_number': 'max_order_number'})
)
orders_agg = orders_agg.merge(user_last_order, on='user_id', how='left')

full_orders = orders_agg.copy()

def unique_categories(df):
    return len(set(cat for cl in df['cart_list'] for cat in cl))

print("\nПолная выборка (full):")
print("  - Пользователей:", full_orders['user_id'].nunique())
print("  - Категорий:", unique_categories(full_orders))

print(full_orders.head(10).to_markdown(index=False))



Полная выборка (full):
  - Пользователей: 20000
  - Категорий: 881
|   user_id | order_completed_at   | cart_list                                                                                                         |   order_number |   max_order_number |
|----------:|:---------------------|:------------------------------------------------------------------------------------------------------------------|---------------:|-------------------:|
|         0 | 2020-07-19 09:59:17  | [20, 82, 441, 57, 14, 405, 430, 379]                                                                              |              1 |                  3 |
|         0 | 2020-08-24 08:55:32  | [133, 5, 26, 10, 382, 14, 22, 41, 25, 441, 411, 799, 432, 84, 83, 383, 409, 821, 405, 402, 57, 396, 379, 82, 157] |              2 |                  3 |
|         0 | 2020-09-02 07:38:25  | [803, 170, 84, 61, 440, 57, 55, 401, 398, 399, 169]                                                               |              3 

In [ ]:
# Блок: Добавление даты без времени и глобального номера заказа
#
# Назначение:
#   - Преобразовать дату и время заказа к дате без времени.
#   - Создать глобальный порядковый номер заказа по дате для всех пользователей.
#
# Вход:
#   - DataFrame full_orders с историей заказов.
# Выход:
#   - DataFrame full_orders с добавленными колонками order_date и global_order_number.
#
full_orders['order_date'] = full_orders['order_completed_at'].dt.date
full_orders = full_orders.sort_values('order_completed_at')

unique_dates = sorted(full_orders['order_date'].unique())
date_to_global_order_num = {date: idx + 1 for idx, date in enumerate(unique_dates)}
full_orders['global_order_number'] = full_orders['order_date'].map(date_to_global_order_num)


In [ ]:
print(full_orders.head().to_markdown(index=False))

|   user_id | order_completed_at   | cart_list                                                                |   order_number |   max_order_number | order_date   |   global_order_number |
|----------:|:---------------------|:-------------------------------------------------------------------------|---------------:|-------------------:|:-------------|----------------------:|
|         2 | 2015-03-22 09:25:46  | [399, 14, 198, 88, 157, 82, 134, 16, 409, 384, 808, 84, 23, 89, 57, 425] |              1 |                 15 | 2015-03-22   |                     1 |
|         3 | 2015-06-18 16:15:33  | [399]                                                                    |              1 |                  7 | 2015-06-18   |                     2 |
|         3 | 2015-07-04 14:05:22  | [399]                                                                    |              2 |                  7 | 2015-07-04   |                     3 |
|         4 | 2015-07-08 06:59:04  | [54, 55]          

In [ ]:
# Создаем колонку с годом и месяцем для каждого заказа
full_orders['year_month'] = full_orders['order_completed_at'].dt.to_period('M')

In [ ]:
print(full_orders.head(10).to_markdown(index=False))

|   user_id | order_completed_at   | cart_list                                                                |   order_number |   max_order_number | order_date   |   global_order_number | year_month   |
|----------:|:---------------------|:-------------------------------------------------------------------------|---------------:|-------------------:|:-------------|----------------------:|:-------------|
|         2 | 2015-03-22 09:25:46  | [399, 14, 198, 88, 157, 82, 134, 16, 409, 384, 808, 84, 23, 89, 57, 425] |              1 |                 15 | 2015-03-22   |                     1 | 2015-03      |
|         3 | 2015-06-18 16:15:33  | [399]                                                                    |              1 |                  7 | 2015-06-18   |                     2 | 2015-06      |
|         3 | 2015-07-04 14:05:22  | [399]                                                                    |              2 |                  7 | 2015-07-04   |                    

In [ ]:
# --- Индивидуальный порядковый номер месяца для каждого пользователя ---
# Получаем уникальные месяцы для каждого пользователя
unique_user_months = (
    full_orders[['user_id', 'year_month']]
    .drop_duplicates()
    .sort_values(['user_id', 'year_month'])
)

# Добавляем порядковый номер месяца для пользователя
unique_user_months['individual_month'] = unique_user_months.groupby('user_id').cumcount() + 1

# Объединяем обратно с исходным датафреймом
full_orders = full_orders.merge(unique_user_months, on=['user_id', 'year_month'], how='left')

In [ ]:
print(full_orders.head(10).to_markdown(index=False))

|   user_id | order_completed_at   | cart_list                                                                |   order_number |   max_order_number | order_date   |   global_order_number | year_month   |   individual_month |
|----------:|:---------------------|:-------------------------------------------------------------------------|---------------:|-------------------:|:-------------|----------------------:|:-------------|-------------------:|
|         2 | 2015-03-22 09:25:46  | [399, 14, 198, 88, 157, 82, 134, 16, 409, 384, 808, 84, 23, 89, 57, 425] |              1 |                 15 | 2015-03-22   |                     1 | 2015-03      |                  1 |
|         3 | 2015-06-18 16:15:33  | [399]                                                                    |              1 |                  7 | 2015-06-18   |                     2 | 2015-06      |                  1 |
|         3 | 2015-07-04 14:05:22  | [399]                                                          

In [ ]:
# --- Глобальный порядковый номер месяца по всему датафрейму ---
# Получаем уникальные глобальные месяцы по дате (без учета пользователя)
unique_global_months = (
    full_orders[['year_month']]
    .drop_duplicates()
    .sort_values('year_month')
    .reset_index(drop=True)
)
unique_global_months['global_month'] = unique_global_months.index + 1

# Объединяем обратно с основным датафреймом
full_orders = full_orders.merge(unique_global_months, on='year_month', how='left')


In [ ]:
print(full_orders.head(10).to_markdown(index=False))

|   user_id | order_completed_at   | cart_list                                                                |   order_number |   max_order_number | order_date   |   global_order_number | year_month   |   individual_month |   global_month |
|----------:|:---------------------|:-------------------------------------------------------------------------|---------------:|-------------------:|:-------------|----------------------:|:-------------|-------------------:|---------------:|
|         2 | 2015-03-22 09:25:46  | [399, 14, 198, 88, 157, 82, 134, 16, 409, 384, 808, 84, 23, 89, 57, 425] |              1 |                 15 | 2015-03-22   |                     1 | 2015-03      |                  1 |              1 |
|         3 | 2015-06-18 16:15:33  | [399]                                                                    |              1 |                  7 | 2015-06-18   |                     2 | 2015-06      |                  1 |              2 |
|         3 | 2015-07-04 14:05:2